# Uncoded Owner Prioritization

Identifies which uncoded newspaper owners/editors would add the most newspaper-years
to the analysis sample if coded. Operational notebook for expanding the dataset.

In [ ]:
import pandas as pd
import sqlite3
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')

DB_PATH = '../data/newspapers.db'
OE_PATH = '../data/personnel_coding/owners_and_editors.csv'
CODE_PATH = '../data/personnel_coding/combined_coding.csv'

# Load intermediate data from 01_data_preparation
df = pd.read_csv('intermediate/analysis_sample.csv')
counts = pd.read_csv('intermediate/sentiment_counts.csv')
paper_year_rr = pd.read_csv('intermediate/paper_year_rr.csv')
person_rr = pd.read_csv('intermediate/person_rr.csv')

print(f'Loaded analysis sample: {len(df)} newspaper-years')

In [ ]:
RAILROAD_COLS = [
    'railroad_stockholder', 'railroad_board_member', 'railroad_company_owner',
    'railroad_donation', 'railroad_professional_services', 'family_connection_railroad'
]


## 9. Highest-Value Uncoded Owners/Editors

Which owners/editors have the most newspaper-years of article data but have not yet been coded for railroad ties?

In [23]:
# Load data
oe = pd.read_csv(OE_PATH)
coding = pd.read_csv(CODE_PATH)
conn = sqlite3.connect(DB_PATH)

# Get newspaper-years that have article data
article_ny = pd.read_sql("""
    SELECT issn, year, COUNT(*) as n_articles
    FROM article_sentiment
    GROUP BY issn, year
""", conn)
conn.close()

# Keep only person rows (non-null person_id)
oe_persons = oe.dropna(subset=['person_id']).copy()
oe_persons['person_id'] = oe_persons['person_id'].astype(int)

# Expand each owner's tenure into individual years
rows = []
for _, r in oe_persons.iterrows():
    for y in range(int(r['first_year']), int(r['last_year']) + 1):
        rows.append({'person_id': r['person_id'], 'name': r['name'],
                      'issn': r['issn'], 'year': y})
person_years = pd.DataFrame(rows)

# Join to article data â€” only count years where we actually have articles
person_articles = person_years.merge(article_ny, on=['issn', 'year'], how='inner')

# Aggregate per person
person_summary = (
    person_articles
    .groupby(['person_id', 'name'])
    .agg(newspaper_years=('year', 'nunique'),
         total_articles=('n_articles', 'sum'),
         newspapers=('issn', 'nunique'))
    .reset_index()
    .sort_values('total_articles', ascending=False)
)

# "Coded" = has at least one non-NaN value in the railroad columns
RAILROAD_COLS = [
    'railroad_stockholder', 'railroad_board_member', 'railroad_company_owner',
    'railroad_donation', 'railroad_professional_services', 'family_connection_railroad'
]
coded_ids = set(
    coding.loc[coding[RAILROAD_COLS].notna().any(axis=1), 'person_id']
    .dropna().astype(int)
)
person_summary['coded'] = person_summary['person_id'].isin(coded_ids)

# Show top 10 by total articles, highlighting coded status
top10 = person_summary.head(35)[['name', 'person_id', 'newspapers', 'newspaper_years', 'total_articles', 'coded']]
print(f"Top 10 owners/editors by article volume (coded: {top10['coded'].sum()}/10)\n")
top10

Top 10 owners/editors by article volume (coded: 28/10)



,name,person_id,newspapers,newspaper_years,total_articles,coded
2,Crosby Noyes,5,1,22,5764,True
12,Lewis Baker,24,4,22,5006,True
11,Harlan P. Hall,22,2,13,4770,True
17,Berry R. Sulgrove,37,1,8,3904,True
16,John C. New,36,1,8,3904,True
0,Whitelaw Reid,3,1,14,2704,True
30,J. H. Estill,71,2,10,2397,True
321,A. C. Cameron,738,1,7,2212,False
24,John F. Morse,52,1,11,1622,True
5,James Gordon Ben- nett,13,1,9,1501,True


In [24]:
# Coded persons: those with at least one non-NaN railroad column
coded_persons = coding[coding[RAILROAD_COLS].notna().any(axis=1)].copy()
coded_persons['railroad_interest'] = (coded_persons[RAILROAD_COLS].fillna(0) > 0).any(axis=1).astype(int)

# Expand coded persons' tenures into person-paper-year rows
oe_coded = oe.dropna(subset=['person_id', 'issn']).copy()
oe_coded['person_id'] = oe_coded['person_id'].astype(int)
oe_coded = oe_coded[oe_coded['person_id'].isin(coded_persons['person_id'])]

coded_tenure = []
for _, row in oe_coded.iterrows():
    for yr in range(int(row['first_year']), int(row['last_year']) + 1):
        coded_tenure.append({'person_id': row['person_id'], 'issn': row['issn'], 'year': yr})
coded_tenure = pd.DataFrame(coded_tenure)

# Load sentiment counts
conn = sqlite3.connect(DB_PATH)
sent = pd.read_sql("""
    SELECT issn, year, labor_sentiment
    FROM article_sentiment
    WHERE issn != '' AND labor_sentiment IS NOT NULL
""", conn)
conn.close()

sent_counts = (
    sent.groupby(['issn', 'year', 'labor_sentiment'])
    .size().unstack(fill_value=0).reset_index()
)
for col in ['anti_labor', 'pro_labor', 'neutral']:
    if col not in sent_counts.columns:
        sent_counts[col] = 0
sent_counts['total_labor'] = sent_counts['anti_labor'] + sent_counts['pro_labor'] + sent_counts['neutral']
sent_counts['anti_labor_intensity'] = sent_counts['anti_labor'] / sent_counts['total_labor']

# Join person-years to sentiment, aggregate per person
person_sent = coded_tenure.merge(sent_counts, on=['issn', 'year'], how='inner')
person_agg = (
    person_sent
    .groupby('person_id')
    .agg(anti_labor=('anti_labor', 'sum'), total_labor=('total_labor', 'sum'))
    .reset_index()
)
person_agg['anti_labor_intensity'] = person_agg['anti_labor'] / person_agg['total_labor']

# Merge with coded person info
result = coded_persons[['person_id', 'name', 'railroad_interest']].merge(
    person_agg[['person_id', 'anti_labor_intensity', 'total_labor']], on='person_id', how='left'
)
result['treated'] = result['railroad_interest'].map({1: 'Yes', 0: 'No'})
result = result.sort_values('anti_labor_intensity', ascending=False)

print(f"Coded persons: {len(result)} ({result['treated'].value_counts().to_dict()})\n")
result[['name', 'treated', 'anti_labor_intensity', 'total_labor']].reset_index(drop=True)

Coded persons: 47 ({'No': 26, 'Yes': 21})



,name,treated,anti_labor_intensity,total_labor
0,Charles M. Stone,Yes,0.588235,306.0
1,Alfales Young,Yes,0.528302,53.0
2,Wm. H. Simpson,Yes,0.517241,58.0
3,General John M. Hedrick | Major Augustus H. Ha...,Yes,0.513514,74.0
4,C. S. Douglas,No,0.500000,4.0
5,John M. Hedrick,Yes,0.500000,64.0
6,C. W. Willard,Yes,0.500000,2.0
7,James Gordon Bennett,Yes,0.423718,1501.0
8,Buckley B. Paddock | Walter Malone,Yes,0.407229,1245.0
9,Whitelaw Reid,Yes,0.406065,2704.0
